# **Implementing Forward and Backward Propagation**

### **Laboratory Exercise 2** - *Deep Learning Fundamentals*
*From the course module:* ***01_Understanding Deep Learning*** *(Laboratory Task 3)*

**Instruction:** Perform a forward and backward propagation in python using the inputs from Laboratory Task 2.

```python
x = np.array([1, 0, 1])
y = np.array([1])

# use relu as the activation function.

# learning rate
lr = 0.001
```

## **Overview**

Neural networks learn through a continuous two-step cycle consisting of forward propagation and backward propagation.

```mermaid
graph LR
    subgraph Forward_Pass [Forward Propagation]
        direction LR
        Input["Input layer"] --> Hidden["Hidden layer"]
        Hidden --> Output["Output layer"]
        Output --> Prediction["Prediction"]
    end

    Prediction --> Loss["Loss Calculation<br>(Compare Prediction vs Target)"]
    Loss --> Backward["Backward Propagation<br>(Compute Gradients via Chain Rule)"]
    Backward --> Update["Weight & Bias Update<br>(Apply Learning Rate)"]
    Update -.->|Next Iteration| Input
```
<br>

During forward propagation, data flows forward from the input layer through the hidden layers to the output layer, where inputs are multiplied by weights, added to biases, and passed through an activation function like ReLU to generate a prediction. 

Following this, backward propagation takes place, where the network evaluates its prediction against the actual target using a loss function. It then works backward from the output to the input layer using the chain rule of calculus to compute gradients, which determine how much each weight and bias contributed to the error. Finally, the network updates its weights and biases using an optimizer and a learning rate to reduce the error in future iterations.

### **Setup**

The network has 3 inputs, 2 hidden layers, and 1 output layer. Every layer uses ReLU. Based on your network configuration table, here are the exact parameters you need to use:

Input:
$$
x = \begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix}
$$   
<br>

Target:
$$
y = 1
$$   
<br>

Hidden-layer weights:
$$
W_h =  \begin{bmatrix} 0.2 & -0.3 \\ 0.4 & 0.1 \\ -0.5 & 0.2 \end{bmatrix}
$$   
<br>

Output-layer weights:
$$
W_o = \begin{bmatrix} -0.3 & -0.2 \end{bmatrix}
$$   
<br>

Biases:
$$
\theta_1 = -0.4,\quad \theta_2 = 0.2,\quad \theta_3 = 0.1
$$   
<br>

Learning Rate:
$$
\eta = 0.001
$$

## **Implementation**

### **Import Libraries**

In [1]:
import numpy as np

### **Define the Inputs, Weights, and Biases**

In [2]:
x = np.array([1, 0, 1], dtype=float)
y = np.array([1], dtype=float)

W_hidden = np.array([
    [ 0.2, -0.3],
    [ 0.4,  0.1],
    [-0.5,  0.2]
], dtype=float)

W_output = np.array([-0.3, -0.2], dtype=float)

theta_hidden = np.array([-0.4, 0.2], dtype=float)
theta_output = 0.1

lr = 0.001

### **Define ReLU and Its Derivative**

Backpropagation needs the derivative of the activation function:

$$
f(Z)=\max(0,Z) \qquad\qquad f'(Z)=\begin{cases}1, & Z>0\\0, & Z\leq 0\end{cases}
$$

In [3]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

### **Forward Propagation**

$$Z_h = xW_h + \theta_h \qquad H = f(Z_h)$$

$$Z_o = H \cdot W_o + \theta_3 \qquad \hat{y} = f(Z_o)$$
<br>

***Hand calculation***

$$Z_1 = (0.2)(1)+(0.4)(0)+(-0.5)(1)-0.4 = -0.7 \;\Rightarrow\; H_1 = 0$$

$$Z_2 = (-0.3)(1)+(0.1)(0)+(0.2)(1)+0.2 = 0.1 \;\Rightarrow\; H_2 = 0.1$$

$$Z_3 = (-0.3)(0)+(-0.2)(0.1)+0.1 = 0.08 \;\Rightarrow\; \hat{y} = 0.08$$

In [ ]:
Z_hidden = x @ W_hidden + theta_hidden
H = relu(Z_hidden)

Z_output = H @ W_output + theta_output
y_hat = relu(Z_output)

E = 0.5 * np.sum((y - y_hat) ** 2)

print("Z_hidden :", Z_hidden)
print("H        :", H)
print("Z_output :", Z_output)
print("y_hat    :", y_hat)
print(f"Error E  : {E:.4f}")

Z_hidden : [-0.7  0.1]
H        : [0.  0.1]
Z_output : 0.08
y_hat    : 0.08
Error E  : 0.4232


### **Backward Propagation**

Using the error $E=\tfrac{1}{2}(y-\hat{y})^2$ and the chain rule, we move from the output back to the inputs.

***Output layer***

$$\frac{\partial E}{\partial \hat{y}} = \hat{y}-y = 0.08 - 1 = -0.92$$

$$\delta_3 = \frac{\partial E}{\partial \hat{y}}\cdot f'(Z_3) = (-0.92)(1) = -0.92$$

$$\frac{\partial E}{\partial W_o} = \delta_3 \cdot H = [(-0.92)(0),\ (-0.92)(0.1)] = [0,\ -0.092]$$

$$\frac{\partial E}{\partial \theta_3} = \delta_3 = -0.92$$
<br>

***Hidden layer***

The error signal is passed back through $W_o$ and gated by the ReLU derivative of each hidden neuron:

$$\delta_h = (W_o \cdot \delta_3)\odot f'(Z_h)$$

$$\delta_1 = (-0.3)(-0.92)\cdot f'(-0.7) = 0.276 \cdot 0 = 0$$

$$\delta_2 = (-0.2)(-0.92)\cdot f'(0.1) = 0.184 \cdot 1 = 0.184$$

$$\frac{\partial E}{\partial W_h} = x^{\top}\delta_h \qquad \frac{\partial E}{\partial \theta_h} = \delta_h$$

In [5]:
dE_dyhat = y_hat - y
delta_output = dE_dyhat * relu_derivative(Z_output)

dW_output = delta_output * H
dtheta_output = delta_output

delta_hidden = (W_output * delta_output) * relu_derivative(Z_hidden)

dW_hidden = np.outer(x, delta_hidden)
dtheta_hidden = delta_hidden

print("delta_output :", delta_output)
print("dW_output    :", dW_output)
print("dtheta_output:", dtheta_output)
print()
print("delta_hidden :", delta_hidden)
print("dW_hidden    :\n", dW_hidden)
print("dtheta_hidden:", dtheta_hidden)

delta_output : [-0.92]
dW_output    : [-0.    -0.092]
dtheta_output: [-0.92]

delta_hidden : [0.    0.184]
dW_hidden    :
 [[0.    0.184]
 [0.    0.   ]
 [0.    0.184]]
dtheta_hidden: [0.    0.184]


### **Update the Weights**

Gradient descent rule with $\eta = 0.001$:

$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \frac{\partial E}{\partial w}$$
<br>

For example, the only output weight that changes is

$$w_{22}^{\text{new}} = -0.2 - (0.001)(-0.092) = -0.199908$$
<br>

Weights connected to $H_1$ do not change, because ReLU output $H_1 = 0$ (dead neuron for this input) and its derivative is $0$.

In [6]:
W_output_new     = W_output     - lr * dW_output
theta_output_new = theta_output - lr * dtheta_output

W_hidden_new     = W_hidden     - lr * dW_hidden
theta_hidden_new = theta_hidden - lr * dtheta_hidden

print("Updated hidden weights:\n", W_hidden_new)
print("Updated hidden biases :", theta_hidden_new)
print("Updated output weights:", W_output_new)
print("Updated output bias   :", theta_output_new)

Updated hidden weights:
 [[ 0.2      -0.300184]
 [ 0.4       0.1     ]
 [-0.5       0.199816]]
Updated hidden biases : [-0.4       0.199816]
Updated output weights: [-0.3      -0.199908]
Updated output bias   : [0.10092]


### **Verify the Update**

After updating, a second forward pass should give a **smaller error** than before.

In [7]:
Z_hidden2 = x @ W_hidden_new + theta_hidden_new
H2 = relu(Z_hidden2)
Z_output2 = H2 @ W_output_new + theta_output_new
y_hat2 = relu(Z_output2)
E2 = 0.5 * np.sum((y - y_hat2) ** 2)

print(f"Before update: y_hat = {float(np.squeeze(y_hat)):.6f}, E = {float(E):.6f}")
print(f"After update : y_hat = {float(np.squeeze(y_hat2)):.6f}, E = {float(E2):.6f}")
print(f"Error decreased: {bool(E2 < E)}")

Before update: y_hat = 0.080000, E = 0.423200
After update : y_hat = 0.081040, E = 0.422244
Error decreased: True


## **Final Answer**

The forward pass results are:

$$
Z_1 = -0.7,\quad H_1 = 0
$$

$$
Z_2 = 0.1,\quad H_2 = 0.1
$$

$$
Z_3 = 0.08,\quad \hat{y} = 0.08
$$
<br>

The target output $y$ is $1$. 

The squared error is:
$$
\boxed{E = 0.4232}
$$
<br>

**Gradients and Parameter Updates**

The gradients computed via backpropagation and their corresponding updated values (using a learning rate of $\eta = 0.001$) are organized below:

| Parameter | Gradient | Updated value |
|---|---|---|
| $w_{11}, w_{13}, w_{15}$ (into $H_1$) | $0$ | $0.2,\ 0.4,\ -0.5$ (unchanged) |
| $w_{12}$ ($x_1 \to H_2$) | $0.184$ | $-0.300184$ |
| $w_{14}$ ($x_2 \to H_2$) | $0$ | $0.1$ (unchanged, since $x_2=0$) |
| $w_{16}$ ($x_3 \to H_2$) | $0.184$ | $0.199816$ |
| $w_{21}$ ($H_1 \to$ output) | $0$ | $-0.3$ (unchanged) |
| $w_{22}$ ($H_2 \to$ output) | $-0.092$ | $-0.199908$ |
| $\theta_1$ | $0$ | $-0.4$ (unchanged) |
| $\theta_2$ | $0.184$ | $0.199816$ |
| $\theta_3$ | $-0.92$ | $0.10092$ |

**Interpretation**

Only the parameters connected to the active path (hidden layer 2 and the output layer) experienced changes, while the parameters associated with hidden layer 1 remained completely untouched because that neuron was silenced by the ReLU activation function ($H_1 = 0$).Because the current prediction ($\hat{y} = 0.08$) sits well below the target value ($y = 1$), the optimization step pushed the output bias upward. Due to the small learning rate ($\eta = 0.001$), the resulting updates are tiny, yielding only a slight decrease in overall error.

**Conclusion**

The network successfully completed its initial backpropagation step, adjusting the active weights and biases to minimize the loss. Because the adjustments are minimal, many subsequent training iterations will be required to bring the prediction $\hat{y}$ close to the target of $1$.